In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import joblib

from sklearn.linear_model import LogisticRegression


# 1. Load Data
data = np.load('../data/fashion_data_complete.npz')

X_train = data['X_train']
y_train = data['y_train']
X_val = data['X_val']
y_val = data['y_val']

# Use only 10,000 training samples for tuning
X_train_small = X_train[:10000]
y_train_small = y_train[:10000]

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 2. Hyperparameter Tuning
n_estimators_list = [50, 100, 200]
max_depth_list = [10, 20, None]

results = {
    'depth_10': [],
    'depth_20': [],
    'depth_None': []
}

print("Starting Random Forest Hyperparameter Tuning...")

for max_depth in max_depth_list:
    key = f"depth_{max_depth}"
    print(f"\n--- Testing max_depth = {max_depth} ---")
    
    for n_estimators in n_estimators_list:
        start = time.time()

        rf_model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=42,
            n_jobs=-1
        )

        rf_model.fit(X_train_small, y_train_small)
        val_acc = rf_model.score(X_val, y_val)
        results[key].append(val_acc)

        end = time.time()
        print(f"n_estimators={n_estimators:3d} | Val Acc: {val_acc:.4f} | Time: {end-start:.2f}s")


# 3. Visualization for Report
plt.figure(figsize=(10, 6))

plt.plot(n_estimators_list, results['depth_10'], marker='o', label='max_depth=10')
plt.plot(n_estimators_list, results['depth_20'], marker='s', label='max_depth=20')
plt.plot(n_estimators_list, results['depth_None'], marker='^', label='max_depth=None')

plt.title('Random Forest Tuning: Number of Trees vs Max Depth')
plt.xlabel('Number of Trees (n_estimators)')
plt.ylabel('Validation Accuracy')
plt.xticks(n_estimators_list)
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()


# 4. Find Best Parameters
best_acc = -1
best_n_estimators = None
best_max_depth = None

for max_depth in max_depth_list:
    key = f"depth_{max_depth}"
    for i, n_estimators in enumerate(n_estimators_list):
        acc = results[key][i]
        if acc > best_acc:
            best_acc = acc
            best_n_estimators = n_estimators
            best_max_depth = max_depth

print(f"\nBest parameters found:")
print(f"n_estimators = {best_n_estimators}")
print(f"max_depth    = {best_max_depth}")
print(f"Best Val Acc = {best_acc:.4f}")

# 5. Train Final Model on Full Training Set
print("\nTraining final Random Forest model on full training set...")

final_rf = RandomForestClassifier(
    n_estimators=best_n_estimators,
    max_depth=best_max_depth,
    random_state=42,
    n_jobs=-1
)

final_rf.fit(X_train, y_train)

# Save model
joblib.dump(final_rf, 'random_forest_fashion_model.joblib')
print("Saved: random_forest_fashion_model.joblib")

In [ ]:
from sklearn.neural_network import MLPClassifier
hidden_sizes = [
    (256,),
    (128, 64),
    (256, 128),
    (128, 128),
    (256, 64),
    (512,),
]
results = []

print("Starting MLP Hyperparameter Tuning...")

for hidden in hidden_sizes:
    start = time.time()

    mlp_model = MLPClassifier(
        hidden_layer_sizes=hidden,
        activation='relu',
        solver='adam',
        alpha=0.0001,
        batch_size=128,
        learning_rate_init=0.001,
        max_iter=30,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=5,
        random_state=42
    )

    mlp_model.fit(X_train_small, y_train_small)
    val_acc = mlp_model.score(X_val, y_val)
    results.append(val_acc)

    end = time.time()
    print(f"Hidden={hidden} | Val Acc: {val_acc:.4f} | Time: {end-start:.2f}s")

# 3. Visualization for Report
labels = [str(h) for h in hidden_sizes]

plt.figure(figsize=(9, 5))
plt.plot(labels, results, marker='o')
plt.title('MLP Tuning: Hidden Layer Size vs Validation Accuracy')
plt.xlabel('Hidden Layer Configuration')
plt.ylabel('Validation Accuracy')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()


# 4. Train Final Model on Full Training Set
best_hidden = hidden_sizes[np.argmax(results)]
print(f"\nBest hidden_layer_sizes found: {best_hidden}")

final_mlp = MLPClassifier(
    hidden_layer_sizes=best_hidden,
    activation='relu',
    solver='adam',
    alpha=0.0001,
    batch_size=128,
    learning_rate_init=0.001,
    max_iter=30,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=5,
    random_state=42
)

print("Training final MLP model on full training set...")
final_mlp.fit(X_train, y_train)

# Save model
joblib.dump(final_mlp, 'mlp_fashion_model.joblib')
print("Saved: mlp_fashion_model.joblib")